## SIMULADOR


In [11]:
## IMPORTS
import  json
from typing import List, Dict, Set
from pathlib import Path

print('[*] Importando os pacotes')


[*] Importando os pacotes


In [12]:
class Automato:
    SIMBOLOS_EPSILON = {''}

    def __init__(self, automato: Dict):
        self.alfabeto = automato['Sigma']
        self.estado_inicial = automato['q0']
        self.estados_finais = automato['F']
        self.transicoes = automato['delta']
        self.tipo_automato = self.classificar_automato()

    def _tem_transicao_epsilon(self) -> bool:
        for estado, trans in self.transicoes.items():
            for situacao in trans:
                if situacao in self.SIMBOLOS_EPSILON:
                    return True
        return False

    def classificar_automato(self) -> str:
        if self._tem_transicao_epsilon():
            return 'AFN'

        for estado, trans in self.transicoes.items():
            for situacao, destinos in trans.items():
                if len(destinos) !=1:
                    return 'AFN'
        return 'AFD'


    def processar_palavra(self, palavra: str) -> bool:
        if(self.tipo_automato == 'AFD'):
            return self.__processar_afd(palavra)
        else:
            return self.__processar_afn(palavra)

    def __processar_afd(self,palavra)-> bool:
        estado_atual  = self.estado_inicial
        print(f"[*] Testando a palavra no AFD {palavra}")
        print('[*] o estado inicial é: ', estado_atual)
        for caractere in palavra:
            if caractere not in self.alfabeto:
                return False

            transicoes_estado = self.transicoes.get(estado_atual, {})

            if caractere not in transicoes_estado:
                return False
            estado_atual = transicoes_estado[caractere][0]
            print(f"[*] Leu {caractere} e foi pro estado: {estado_atual}")
        print('[*] teste finalizado')
        resultado_teste= True if estado_atual in self.estados_finais else False
        return resultado_teste

    def _fecho_epsilon(self, estados: Set[str]) -> Set[str]:
        fecho: Set[str] = set(estados)
        pilha = list(estados)

        while pilha:
            estado = pilha.pop()
            transicoes_estado = self.transicoes.get(estado, {})
            for simbolo in self.SIMBOLOS_EPSILON:
                for destino in transicoes_estado.get(simbolo, []):
                    if destino not in fecho:
                        fecho.add(destino)
                        pilha.append(destino)

        return fecho

    def __processar_afn(self, palavra: str) -> bool:
        estados_atuais: Set[str] = self._fecho_epsilon({self.estado_inicial})

        print(f"[*] Testando a palavra no AFN: '{palavra}'")
        print(f"[*] Os estados iniciais (com fecho-épsilon) são: {estados_atuais}")

        for caractere in palavra:
            if caractere not in self.alfabeto:
                print(f"[-] Caractere '{caractere}' não pertence ao alfabeto. Palavra rejeitada.")
                return False

            proximos_estados: Set[str] = set()

            for estado in estados_atuais:
                transicoes_estado = self.transicoes.get(estado, {})
                if caractere in transicoes_estado:
                    proximos_estados.update(transicoes_estado[caractere])

            estados_atuais = self._fecho_epsilon(proximos_estados)
            print(f"[*] Leu '{caractere}' e ramificou para os estados (com fecho-épsilon): {estados_atuais}")

            if not estados_atuais:
                print("[-] Caminho sem saída encontrado (conjunto de estados vazio). Palavra rejeitada.")
                return False

        print('[*] Teste finalizado')
        resultado_teste = any(estado in self.estados_finais for estado in estados_atuais)

        return resultado_teste

    def rodar_testes(self, palavrasTestes: List[str]):
        print('[*] Testando palavras testes')
        for palavra in palavrasTestes:
            resultado = self.processar_palavra(palavra)
            print(f"[*] {palavra}: {resultado}")
            print('-----------------------------------------------------')

In [13]:
def carregar_automatos():
    path_data = Path('./data')
    automatos = []

    for arquivo in path_data.glob('*.json'):
        with open(arquivo) as json_file:
            try:
                dados = json.load(json_file)
                automatos.append((arquivo.name, dados))
                print(f"Lido com sucesso: {arquivo.name}")
            except json.JSONDecodeError:
                print(f"Erro de formatação no arquivo: {arquivo.name}")

    print("[*] Tudo lido")
    return automatos

In [14]:
def escolher_automato(automatos):
    print("\n--- Automatos disponíveis ---")
    for i, (nome, _, _) in enumerate(automatos, 1):
        print(f"  {i}. {nome}")

    try:
        num = int(input("Digite o número do automato: "))
        if 1 <= num <= len(automatos):
            return automatos[num - 1]
        print(f"[-] Digite um número entre 1 e {len(automatos)}.")
    except ValueError:
        print("[-] Entrada inválida, digite um número.")
    return None



In [15]:

def menu():
    automatos = [
        (nome, Automato(dados), dados)
        for nome, dados in carregar_automatos()
    ]
    if not automatos:
        print("[-] Nenhum automato encontrado em ./data")
        return

    while True:
        print("\n========= MENU =========")
        print("1. Rodar automato")
        print("2. Testar todos os automatos")
        print("3. Testar uma palavra")
        print("4. Sair")
        opcao = input("Escolha uma opção: ").strip()

        if opcao == '1':
            escolha = escolher_automato(automatos)
            if escolha:
                nome, automato, dados = escolha
                print(f"\n[*] Rodando testes de: {nome}")
                automato.rodar_testes(dados.get("palavras_teste", []))

        elif opcao == '2':
            for nome, automato, dados in automatos:
                print(f"\n[*] === {nome} ===")
                automato.rodar_testes(dados.get("palavras_teste", []))

        elif opcao == '3':
            escolha = escolher_automato(automatos)
            if escolha:
                nome, automato, _ = escolha
                palavra = input("Digite a palavra a ser testada: ")
                resultado = automato.processar_palavra(palavra)
                status = "ACEITA" if resultado else "REJEITADA"
                print(f"\n[*] Palavra '{palavra}' foi {status} pelo automato {nome}")

        elif opcao == '4':
            print("[*] Saindo...")
            break

        else:
            print("[-] Opção inválida.")


In [17]:
menu()

Lido com sucesso: AFD_maior.json
Lido com sucesso: AFD_simples.json
Lido com sucesso: AFn_maior.json
Lido com sucesso: AFN_simples.json
Lido com sucesso: AFN_epsilon.json
[*] Tudo lido

========= MENU =========
1. Rodar automato
2. Testar todos os automatos
3. Testar uma palavra
4. Sair

--- Automatos disponíveis ---
  1. AFD_maior.json
  2. AFD_simples.json
  3. AFn_maior.json
  4. AFN_simples.json
  5. AFN_epsilon.json
[-] Entrada inválida, digite um número.

========= MENU =========
1. Rodar automato
2. Testar todos os automatos
3. Testar uma palavra
4. Sair
[*] Saindo...
